# Joshi Part 7: Convergence Analysis and Variance Reduction

Based on **"The Concepts and Practice of Mathematical Finance"** by Mark S. Joshi.

This notebook covers Joshi's SimpleMC7 (ConvergenceTable) and variance reduction:

1. **Convergence tracking** - How MC estimates improve with more paths
2. **Standard error and confidence intervals**
3. **Antithetic variates** - Variance reduction via $Z$ and $-Z$
4. **Control variates** - Using a correlated known-price instrument

The central limit theorem tells us MC error is $O(1/\sqrt{N})$.

In [ ]:
import math
import random
from RustQuant.stochastics import GeometricBrownianMotion
from RustQuant.instruments import BlackScholesMerton, OptionType

spot = 100.0
strike = 100.0
rate = 0.05
vol = 0.20
T = 1.0

# Exact price for reference
bsm = BlackScholesMerton(
    underlying_price=spot, strike_price=strike, volatility=vol,
    risk_free_rate=rate, cost_of_carry=rate,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Call,
)
exact_price = bsm.price()
print(f"Exact BS Call Price: {exact_price:.6f}")

## 1. Convergence Table (Joshi's SimpleMC7)

Joshi's ConvergenceTable tracks the running mean, standard error, and
confidence interval as paths accumulate. We snapshot at powers of 2.

In [ ]:
class ConvergenceTable:
    """Joshi's ConvergenceTable: track running statistics."""
    def __init__(self):
        self.sum = 0.0
        self.sum_sq = 0.0
        self.count = 0
        self.snapshots = []
    
    def add(self, value):
        self.sum += value
        self.sum_sq += value * value
        self.count += 1
    
    @property
    def mean(self):
        return self.sum / self.count
    
    @property
    def variance(self):
        return self.sum_sq / self.count - self.mean**2
    
    @property
    def std_error(self):
        return math.sqrt(self.variance / self.count)
    
    def snapshot(self):
        se = self.std_error
        self.snapshots.append({
            'n': self.count,
            'mean': self.mean,
            'std_error': se,
            'ci_95': 1.96 * se,
        })

In [ ]:
random.seed(42)
table = ConvergenceTable()
total_paths = 2**20  # ~1M paths
next_snap = 256
df = math.exp(-rate * T)

for _ in range(total_paths):
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
    s_t = spot * math.exp((rate - 0.5 * vol**2) * T + vol * math.sqrt(T) * z)
    table.add(df * max(s_t - strike, 0.0))
    
    if table.count == next_snap:
        table.snapshot()
        next_snap *= 2

table.snapshot()

print(f"{'Paths':>10} {'Price':>10} {'Std Err':>10} {'95% CI ±':>10} {'Error':>10}")
print("-" * 50)
for s in table.snapshots:
    err = s['mean'] - exact_price
    print(f"{s['n']:>10} {s['mean']:>10.4f} {s['std_error']:>10.6f} {s['ci_95']:>10.6f} {err:>+10.4f}")

## 2. Antithetic Variates

**Idea (Joshi):** For each random draw $Z$, also compute the payoff with $-Z$.
Since $\text{Cov}(f(Z), f(-Z)) < 0$ for convex payoffs, the variance of the
average is reduced.

$$\hat{C}_{\text{AV}} = \frac{1}{2}\left[f(Z) + f(-Z)\right]$$

In [ ]:
random.seed(42)
table_av = ConvergenceTable()
next_snap = 256

for _ in range(total_paths):
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
    
    # Path with Z
    s_pos = spot * math.exp((rate - 0.5 * vol**2) * T + vol * math.sqrt(T) * z)
    # Antithetic path with -Z
    s_neg = spot * math.exp((rate - 0.5 * vol**2) * T + vol * math.sqrt(T) * (-z))
    
    # Average payoff
    payoff_av = 0.5 * (max(s_pos - strike, 0) + max(s_neg - strike, 0))
    table_av.add(df * payoff_av)
    
    if table_av.count == next_snap:
        table_av.snapshot()
        next_snap *= 2

table_av.snapshot()

print(f"{'Paths':>10} {'Price':>10} {'Std Err':>10} {'95% CI ±':>10} {'Error':>10}")
print("-" * 50)
for s in table_av.snapshots:
    err = s['mean'] - exact_price
    print(f"{s['n']:>10} {s['mean']:>10.4f} {s['std_error']:>10.6f} {s['ci_95']:>10.6f} {err:>+10.4f}")

## 3. Variance Reduction Comparison

In [ ]:
# Compare standard errors at matching path counts
print("Variance Reduction Comparison:")
print(f"{'Paths':>10} {'Std MC SE':>12} {'AV SE':>12} {'Ratio':>8}")
print("-" * 42)

for s_std, s_av in zip(table.snapshots, table_av.snapshots):
    if s_std['n'] == s_av['n']:
        ratio = s_std['std_error'] / s_av['std_error'] if s_av['std_error'] > 0 else float('inf')
        print(f"{s_std['n']:>10} {s_std['std_error']:>12.6f} {s_av['std_error']:>12.6f} {ratio:>8.2f}x")

print("\n-> Antithetic variates reduce std error by ~30-50% (effectively ~2x more paths)")

## 4. Control Variates

**Idea:** Use a correlated instrument with a known price to reduce variance.
For Asian options, the geometric average Asian has a closed-form price and
is highly correlated with the arithmetic average version.

$$\hat{C}_{\text{CV}} = \hat{C}_{\text{arith}} - \beta(\hat{C}_{\text{geo}} - C_{\text{geo}}^{\text{exact}})$$

Here we demonstrate the concept with a vanilla call as control for a slightly
out-of-the-money call (strike = 105).

In [ ]:
# Control variate: use ATM call (known price) to improve OTM call estimate
strike_otm = 105.0

bsm_atm = BlackScholesMerton(
    underlying_price=spot, strike_price=strike, volatility=vol,
    risk_free_rate=rate, cost_of_carry=rate,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Call,
)
exact_atm = bsm_atm.price()

# Generate shared random paths
random.seed(123)
N = 50_000

payoffs_otm = []
payoffs_atm = []

for _ in range(N):
    u1, u2 = random.random(), random.random()
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    s_t = spot * math.exp((rate - 0.5*vol**2)*T + vol*math.sqrt(T)*z)
    payoffs_otm.append(df * max(s_t - strike_otm, 0))
    payoffs_atm.append(df * max(s_t - strike, 0))

# Standard estimate
mean_otm = sum(payoffs_otm) / N
mean_atm = sum(payoffs_atm) / N

# Compute optimal beta
cov_sum = sum((a - mean_atm) * (o - mean_otm) for a, o in zip(payoffs_atm, payoffs_otm))
var_atm = sum((a - mean_atm)**2 for a in payoffs_atm)
beta = cov_sum / var_atm

# Control variate estimate
cv_payoffs = [o - beta * (a - exact_atm) for o, a in zip(payoffs_otm, payoffs_atm)]
mean_cv = sum(cv_payoffs) / N

# Standard errors
se_std = math.sqrt(sum((p - mean_otm)**2 for p in payoffs_otm) / N / N)
se_cv = math.sqrt(sum((p - mean_cv)**2 for p in cv_payoffs) / N / N)

bsm_otm = BlackScholesMerton(
    underlying_price=spot, strike_price=strike_otm, volatility=vol,
    risk_free_rate=rate, cost_of_carry=rate,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Call,
)

print(f"OTM Call (K={strike_otm}):")
print(f"  Exact BS:        {bsm_otm.price():.4f}")
print(f"  Standard MC:     {mean_otm:.4f}  (SE: {se_std:.6f})")
print(f"  Control Variate: {mean_cv:.4f}  (SE: {se_cv:.6f})")
print(f"  Beta:            {beta:.4f}")
print(f"  Variance ratio:  {se_std/se_cv:.2f}x improvement")

## Summary

| Technique | Variance Reduction | Cost |
|-----------|-------------------|------|
| Standard MC | Baseline | $N$ paths |
| Antithetic Variates | ~2x | Same $N$ (paired paths) |
| Control Variates | Variable (depends on correlation) | Need known-price instrument |

**Key insight (Joshi):** MC error is $O(1/\sqrt{N})$. To halve the error, you need
4x the paths. Variance reduction techniques effectively give you "free" paths.